# Trabajo Práctico: Búsqueda Voraz y A* sobre un Grafo Dirigido
Nombre: Javier Atiencia

## 1. y 2. Definición del Grafo y Estructuras de Datos
En la siguiente celda se definen las aristas, costos, heurísticas y la función inicial para la creación de nodos.

In [1]:
import heapq

# 1. El grafo dirigido (aristas y costos)
aristas = {
    'S': {'A': 2, 'B': 2},
    'A': {'C': 2, 'D': 5},
    'B': {'D': 2},
    'C': {'G': 3},
    'D': {'G': 6},
    'G': {}
}

# Heurística (estimación del costo restante hasta G)
h_n = {
    'S': 7,
    'A': 5,
    'B': 7,
    'C': 3,
    'D': 6,
    'G': 0
}

# 2. Estructura de datos
def crear_nodo(estado, padre=None, accion=None, g=0, h=0):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "g": g,
        "h": h,
        "f": g + h,
    }

## 3. Requerimientos de implementación
La siguiente función `busqueda` implementa el esqueleto común. Al cambiar la función de prioridad, permite ejecutar UCS, Voraz o A*. Incluye recolección de métricas y la traza de ejecución.

In [2]:
def busqueda(problema, heuristica, algoritmo="UCS"):
    inicio, objetivo, grafo = problema
    contador_insercion = 0
    frontera = []
    mejor_g = {}
    
    if algoritmo == "UCS":
        prioridad = lambda g, h: g
    elif algoritmo == "Voraz":
        prioridad = lambda g, h: h
    elif algoritmo == "A*":
        prioridad = lambda g, h: g + h
    else:
        raise ValueError("Algoritmo desconocido")

    h_inicial = heuristica[inicio]
    nodo_inicial = crear_nodo(inicio, g=0, h=h_inicial)
    nodo_inicial['f'] = prioridad(0, h_inicial)
    
    mejor_g[inicio] = 0
    heapq.heappush(frontera, (nodo_inicial['f'], contador_insercion, nodo_inicial))
    contador_insercion += 1
    
    estados_generados = 1
    estados_expandidos = 0
    frontera_maxima = 1
    reaperturas = 0
    
    print(f"\n{'='*40}")
    print(f"--- Ejecutando Búsqueda: {algoritmo} ---")
    print(f"{'='*40}")
    
    while frontera:
        frontera_maxima = max(frontera_maxima, len(frontera))
        frontera_str = ", ".join([f"{n['estado']}(f={p})" for p, _, n in sorted(frontera)])
        
        f_actual, _, nodo = heapq.heappop(frontera)
        estado = nodo['estado']
        
        print(f"\n> Extraemos: {estado} (f={f_actual})")
        print(f"  Frontera antes de extraer: [{frontera_str}]")
        
        if nodo['g'] > mejor_g.get(estado, float('inf')):
            print(f"  [!] Nodo {estado} es obsoleto. Descartamos.")
            continue
            
        if estado == objetivo:
            print(f"  [★] ¡Objetivo alcanzado!")
            camino = []
            actual = nodo
            while actual:
                camino.append(actual['estado'])
                actual = actual['padre']
            camino.reverse()
            return {
                "Algoritmo": algoritmo,
                "Camino": " -> ".join(camino),
                "Costo": nodo['g'],
                "Generados": estados_generados,
                "Expandidos": estados_expandidos,
                "FronteraMax": frontera_maxima,
                "Reaperturas": reaperturas
            }
            
        estados_expandidos += 1
            
        for sucesor, costo_arista in grafo.get(estado, {}).items():
            nuevo_g = nodo['g'] + costo_arista
            
            if nuevo_g < mejor_g.get(sucesor, float('inf')):
                if sucesor in mejor_g:
                    reaperturas += 1
                    print(f"  [Reapertura] Encontramos un mejor camino hacia {sucesor} (nuevo_g: {nuevo_g})")
                    
                mejor_g[sucesor] = nuevo_g
                h_sucesor = heuristica[sucesor]
                f_sucesor = prioridad(nuevo_g, h_sucesor)
                
                nuevo_nodo = crear_nodo(estado=sucesor, padre=nodo, accion=f"{estado} -> {sucesor}", g=nuevo_g, h=h_sucesor)
                nuevo_nodo['f'] = f_sucesor
                
                heapq.heappush(frontera, (f_sucesor, contador_insercion, nuevo_nodo))
                contador_insercion += 1
                estados_generados += 1
                print(f"  + Insertamos sucesor: {sucesor} (f={f_sucesor})")
                
    return "Fracaso" 

## 5. Verificación y comparación
1. Reproducción de la traza completa de UCS, voraz y A* sobre el grafo.

In [3]:
import pandas as pd
problema = ('S', 'G', aristas)

resultados_ucs = busqueda(problema, h_n, "UCS")
resultados_voraz = busqueda(problema, h_n, "Voraz")
resultados_a_star = busqueda(problema, h_n, "A*")

df_resultados = pd.DataFrame([resultados_ucs, resultados_voraz, resultados_a_star])
df_resultados.set_index("Algoritmo", inplace=True)



--- Ejecutando Búsqueda: UCS ---

> Extraemos: S (f=0)
  Frontera antes de extraer: [S(f=0)]
  + Insertamos sucesor: A (f=2)
  + Insertamos sucesor: B (f=2)

> Extraemos: A (f=2)
  Frontera antes de extraer: [A(f=2), B(f=2)]
  + Insertamos sucesor: C (f=4)
  + Insertamos sucesor: D (f=7)

> Extraemos: B (f=2)
  Frontera antes de extraer: [B(f=2), C(f=4), D(f=7)]
  [Reapertura] Encontramos un mejor camino hacia D (nuevo_g: 4)
  + Insertamos sucesor: D (f=4)

> Extraemos: C (f=4)
  Frontera antes de extraer: [C(f=4), D(f=4), D(f=7)]
  + Insertamos sucesor: G (f=7)

> Extraemos: D (f=4)
  Frontera antes de extraer: [D(f=4), D(f=7), G(f=7)]

> Extraemos: D (f=7)
  Frontera antes de extraer: [D(f=7), G(f=7)]
  [!] Nodo D es obsoleto. Descartamos.

> Extraemos: G (f=7)
  Frontera antes de extraer: [G(f=7)]
  [★] ¡Objetivo alcanzado!

--- Ejecutando Búsqueda: Voraz ---

> Extraemos: S (f=7)
  Frontera antes de extraer: [S(f=7)]
  + Insertamos sucesor: A (f=5)
  + Insertamos sucesor: B (f=7)


2. Completar la tabla comparativa:

| Resultado | UCS | Voraz | A* |
| :--- | :--- | :--- | :--- |
| **Camino** | S -> A -> C -> G | S -> A -> C -> G | S -> A -> C -> G |
| **Costo** | 7 | 7 | 7 |
| **Prioridad** | g | h | g+h |
| **Expandidos antes de extraer G** | 5 | 3 | 3 |
| **Estados Generados** | 7 | 6 | 6 |
| **Frontera Máxima** | 3 | 3 | 3 |
| **Reaperturas** | 1 | 0 | 0 |

## 6. Preguntas de Análisis
**1. ¿Por qué voraz y A* coinciden en este grafo? ¿Qué condición del grafo y de la heurística lo explica?**
Coinciden devolviendo el mismo camino (S -> A -> C -> G) porque la heurística proporcionada guía perfectamente hacia la meta sin introducir engaños (óptimos locales costosos). Desde 'S', la heurística de 'A' (5) es mejor que la de 'B' (7), por lo que Voraz elige ir hacia 'A', y desde allí la heurística sigue disminuyendo (C=3, G=0) en el que casualmente es el camino de menor costo real. Es decir, coinciden porque en este grafo particular la heurística subestima de manera muy precisa la ruta óptima.

**2. ¿Garantiza voraz devolver el camino de menor costo en general? Justificá con la propiedad de su prioridad (no con este ejemplo).**
No, Voraz (Greedy) NO garantiza optimalidad. Al basar su prioridad únicamente en `h` (la estimación de lo que falta) e ignorar el costo acumulado `g`, puede verse tentado a tomar un nodo que parece estar muy cerca de la meta, pero cuyo costo para llegar hasta él fue altísimo. Solo busca acercarse visualmente al objetivo más rápido, pero no evalúa el costo total del viaje.

**3. ¿Qué ocurre si se usa `h = 0` en A*? ¿Con qué algoritmo coincide entonces?**
Si la heurística es siempre 0, la función de prioridad de A* (`f = g + h`) se convierte en `f = g + 0`, es decir, `f = g`. Por lo tanto, A* se degenera y coincide exactamente con el comportamiento de **Costo Uniforme (UCS)**.

**4. ¿Hubo reaperturas en UCS? ¿Y en A* y voraz? ¿Por qué?**
- **UCS:** **Sí, hubo 1 reapertura**. Al expandir 'A', se generó el sucesor 'D' con un costo de 7 (S->A->D). Luego, al expandir 'B', se volvió a generar 'D' pero con un costo mejor de 4 (S->B->D). Esto provocó que se reabriera y se actualizara el mejor camino conocido hacia 'D'.
- **Voraz:** **No hubo**. Fue directo al objetivo sin tener que retroceder a caminos descartados. (Igualmente Voraz no maneja óptimos, se queda con lo primero que encuentra).
- **A*:** **No hubo**. No hubo reaperturas porque la heurística del grafo es *consistente* (monótona). En A*, cuando la heurística es consistente, el primer camino encontrado hacia cualquier estado extraído de la frontera está garantizado de ser el más barato, evitando reaperturas.

**5. ¿"Expandir menos estados" significa "camino más barato"? Relacionalo con lo que muestran UCS y voraz aquí.**
No, "expandir menos estados" habla de **eficiencia de tiempo** (se revisó menos porción del mapa), mientras que un "camino más barato" habla de **optimalidad** (la calidad de la solución encontrada).
En este ejemplo, Voraz expandió menos estados (3) que UCS (5) y aun así encontró un camino igual de barato (costo 7). Sin embargo, esto fue solo suerte gracias a cómo estaban dispuestos los valores de `h`. Si Voraz se hubiera engañado, podría haber expandido muy pocos estados pero devolver un camino carísimo, mientras que UCS siempre expandirá más estados para asegurar encontrar el camino más barato matemáticamente.